In [1]:
!pip install transformers datasets scikit-learn accelerate -q

In [2]:
import torch, json, os, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

if not os.path.exists('katfishnet'):
    os.system('git clone https://github.com/Shinwoo-Park/detecting_llm_generated_korean_text_through_linguistic_analysis.git katfishnet')

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

essay    = load_jsonl('katfishnet/katfish_dataset/essay.jsonl')
abstract = load_jsonl('katfishnet/katfish_dataset/abstract.jsonl')
poetry   = load_jsonl('katfishnet/katfish_dataset/poetry.jsonl')

df_essay    = pd.DataFrame(essay);    df_essay['genre']    = 'essay'
df_abstract = pd.DataFrame(abstract); df_abstract['genre'] = 'abstract'
df_poetry   = pd.DataFrame(poetry);   df_poetry['genre']   = 'poetry'
df_all = pd.concat([df_essay, df_abstract, df_poetry]).reset_index(drop=True)
print(f"전체: {len(df_all)}개")
print(df_all['written_by'].value_counts())

디바이스: cuda
GPU: NVIDIA A100-SXM4-80GB
전체: 2094개
written_by
human                 470
gpt-4o-2024-05-13     470
solar-1-mini-chat     429
qwen2:72b-instruct    387
llama3.1:70b          338
Name: count, dtype: int64


In [3]:
def compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device, max_length=256):
    inp_o = tok_obs(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        loss_o = m_obs(**inp_o, labels=inp_o['input_ids']).loss.item()

    inp_p  = tok_per(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    inp_o2 = tok_obs(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        logits_p = m_per(**inp_p).logits
        logits_o = m_obs(**inp_o2).logits

    seq_len = min(logits_p.shape[1], logits_o.shape[1]) - 1
    if seq_len <= 0:
        return 0.0
    lp = logits_p[:, :seq_len, :]
    lo = logits_o[:, :seq_len, :]
    v  = min(lp.shape[-1], lo.shape[-1])
    lp, lo = lp[..., :v], lo[..., :v]
    cross_loss = -(torch.softmax(lp, dim=-1) * torch.log_softmax(lo, dim=-1)).sum(-1).mean().item()
    return loss_o / (cross_loss + 1e-10)

def find_threshold(scores, labels, target_fpr=0.05):
    thresholds = np.linspace(scores.min(), scores.max(), 1000)
    best_thresh, best_recall = None, 0
    for t in thresholds:
        preds = (scores >= t).astype(int)
        fp = ((preds==1) & (labels==0)).sum()
        tn = ((preds==0) & (labels==0)).sum()
        tp = ((preds==1) & (labels==1)).sum()
        fn = ((preds==0) & (labels==1)).sum()
        fpr    = fp / (fp + tn + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        if fpr <= target_fpr and recall > best_recall:
            best_recall = recall
            best_thresh = t
    return best_thresh, best_recall

def run_ood_experiment(model_name, m_obs, tok_obs, m_per, tok_per, human_test, llm_groups, device):
    """OOD 실험 전체 실행 + 장르별 분석"""
    print(f"\n{'='*60}")
    print(f"실험: {model_name}")
    print(f"{'='*60}")

    # Human 점수
    print("Human 점수 계산 중...")
    human_scores = []
    for text in tqdm(human_test['text'].tolist()):
        human_scores.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
    human_scores = np.array(human_scores)

    # LLM 점수 저장
    llm_scores_dict = {}
    for llm_name, llm_df in llm_groups.items():
        print(f"{llm_name} 점수 계산 중...")
        scores = []
        for text in tqdm(llm_df['text'].tolist()):
            scores.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
        llm_scores_dict[llm_name] = {
            'scores': np.array(scores),
            'genres': llm_df['genre'].tolist()
        }

    # 전체 AUC + Recall
    print(f"\n----- {model_name} 전체 결과 -----")
    print(f"{'LLM':<12} {'AUC':>8} {'R@FPR5%':>10} {'R@FPR10%':>10}")
    print("-" * 45)
    aucs, r5s, r10s = [], [], []
    for llm_name in llm_groups.keys():
        l_sc = llm_scores_dict[llm_name]['scores']
        sc   = np.concatenate([human_scores, l_sc])
        lb   = np.array([0]*len(human_scores) + [1]*len(l_sc))
        auc  = roc_auc_score(lb, -sc)
        _, r5  = find_threshold(-sc, lb, target_fpr=0.05)
        _, r10 = find_threshold(-sc, lb, target_fpr=0.10)
        aucs.append(auc*100); r5s.append(r5*100); r10s.append(r10*100)
        print(f"{llm_name:<12} {auc*100:>8.2f} {r5*100:>10.2f} {r10*100:>10.2f}")
    print("-" * 45)
    print(f"{'평균':<12} {np.mean(aucs):>8.2f} {np.mean(r5s):>10.2f} {np.mean(r10s):>10.2f}")

    # 장르별 AUC
    print(f"\n----- {model_name} 장르별 AUC -----")
    print(f"{'장르':<12} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균':>8}")
    print("-" * 45)
    for genre in ['essay', 'abstract', 'poetry']:
        h_genre = human_test[human_test['genre'] == genre]
        h_idx   = [i for i, t in enumerate(human_test['text'].tolist())
                   if t in set(h_genre['text'].tolist())]
        h_sc    = human_scores[h_idx]
        row = []
        for llm_name in ['Solar', 'Qwen2', 'Llama3.1']:
            data  = llm_scores_dict[llm_name]
            l_idx = [i for i, g in enumerate(data['genres']) if g == genre]
            l_sc  = data['scores'][l_idx]
            if len(l_sc) == 0:
                row.append(0); continue
            sc_all = np.concatenate([h_sc, l_sc])
            lb_all = np.array([0]*len(h_sc) + [1]*len(l_sc))
            row.append(roc_auc_score(lb_all, -sc_all) * 100)
        print(f"{genre:<12} {row[0]:>8.2f} {row[1]:>8.2f} {row[2]:>10.2f} {np.mean(row):>8.2f}")

    return {'human_scores': human_scores, 'llm_scores_dict': llm_scores_dict}

print('함수 정의 완료')

함수 정의 완료


In [4]:
human_df   = df_all[df_all['label'] == 0]
human_test = human_df.sample(frac=0.2, random_state=42)
print(f"Human 테스트셋: {len(human_test)}개")

llm_groups = {
    'Solar':    df_all[df_all['written_by'].str.contains('solar',  case=False, na=False)],
    'Qwen2':    df_all[df_all['written_by'].str.contains('qwen',   case=False, na=False)],
    'Llama3.1': df_all[df_all['written_by'].str.contains('llama',  case=False, na=False)],
}
for name, grp in llm_groups.items():
    print(f"{name}: {len(grp)}개")

Human 테스트셋: 94개
Solar: 429개
Qwen2: 387개
Llama3.1: 338개


In [ ]:
print("Falcon-7B (base) 로드 중...")
tok_f1 = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")
m_f1   = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b", torch_dtype=torch.float16, device_map="auto")
m_f1.eval()

print("Falcon-7B-instruct 로드 중...")
tok_f2 = AutoTokenizer.from_pretrained("tiiuae/falcon-7b-instruct")
m_f2   = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b-instruct", torch_dtype=torch.float16, device_map="auto")
m_f2.eval()

result_falcon = run_ood_experiment(
    "Falcon-7B + Falcon-7B-instruct (원본 Binoculars)",
    m_f1, tok_f1, m_f2, tok_f2,
    human_test, llm_groups, device
)

Falcon-7B (base) 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Qwen2 점수 계산 중...


100%|██████████| 387/387 [00:57<00:00,  6.69it/s]


Llama3.1 점수 계산 중...


100%|██████████| 338/338 [00:49<00:00,  6.85it/s]



----- Falcon-7B + Falcon-7B-instruct (원본 Binoculars) 전체 결과 -----
LLM               AUC    R@FPR5%   R@FPR10%
---------------------------------------------
Solar           62.83      13.99      19.81
Qwen2           46.79       2.58       7.75
Llama3.1        56.18       5.33       8.88
---------------------------------------------
평균              55.27       7.30      12.15

----- Falcon-7B + Falcon-7B-instruct (원본 Binoculars) 장르별 AUC -----
장르             →Solar   →Qwen2  →Llama3.1       평균
---------------------------------------------
essay           75.26    56.25      71.12    67.54
abstract        50.38    40.72      31.90    41.00
poetry          72.05    52.87      74.51    66.48


In [ ]:
# Falcon 메모리 해제
del m_f1, m_f2
torch.cuda.empty_cache()
print("Falcon 해제 완료")

print("Qwen2.5-7B (base) 로드 중...")
tok_q1 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
m_q1   = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B", torch_dtype=torch.float16, device_map="auto")
m_q1.eval()

print("Qwen2.5-7B-Instruct 로드 중...")
tok_q2 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
m_q2   = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct", torch_dtype=torch.float16, device_map="auto")
m_q2.eval()

result_qwen7b = run_ood_experiment(
    "Qwen2.5-7B + Qwen2.5-7B-Instruct",
    m_q1, tok_q1, m_q2, tok_q2,
    human_test, llm_groups, device
)

Falcon 해제 완료
Qwen2.5-7B (base) 로드 중...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Qwen2 점수 계산 중...



100%|██████████| 387/387 [00:45<00:00,  8.59it/s]


Llama3.1 점수 계산 중...



100%|██████████| 338/338 [00:38<00:00,  8.75it/s]



----- Qwen2.5-7B + Qwen2.5-7B-Instruct 전체 결과 -----
LLM               AUC    R@FPR5%   R@FPR10%
---------------------------------------------
Solar           86.64      70.40      76.46
Qwen2           80.31      24.03      47.03
Llama3.1        82.61      47.63      61.24
---------------------------------------------
평균              83.19      47.35      61.58

----- Qwen2.5-7B + Qwen2.5-7B-Instruct 장르별 AUC -----
장르             →Solar   →Qwen2  →Llama3.1       평균
---------------------------------------------
essay           98.95    89.30      93.13    93.79
abstract        64.23    32.13      58.89    51.75
poetry          93.74    74.30      85.75    84.60


In [ ]:
# Qwen7B 메모리 해제
# del m_q1, m_q2
# torch.cuda.empty_cache()
# print("Qwen7B 해제 완료")

print("EXAONE-3.5-2.4B-Instruct 로드 중...")
tok_e1 = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")
m_e1   = AutoModelForCausalLM.from_pretrained(
    "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct", torch_dtype=torch.float16, device_map="auto")
m_e1.eval()

print("EXAONE-3.5-7.8B-Instruct 로드 중...")
tok_e2 = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct")
m_e2   = AutoModelForCausalLM.from_pretrained(
    "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct", torch_dtype=torch.float16, device_map="auto")
m_e2.eval()

result_exaone = run_ood_experiment(
    "EXAONE-3.5-2.4B + EXAONE-3.5-7.8B",
    m_e1, tok_e1, m_e2, tok_e2,
    human_test, llm_groups, device
)

EXAONE-3.5-2.4B-Instruct 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct .
 You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Qwen2 점수 계산 중...


100%|██████████| 387/387 [00:46<00:00,  8.26it/s]


Llama3.1 점수 계산 중...


100%|██████████| 338/338 [00:40<00:00,  8.39it/s]



----- EXAONE-3.5-2.4B + EXAONE-3.5-7.8B 전체 결과 -----
LLM               AUC    R@FPR5%   R@FPR10%
---------------------------------------------
Solar           85.03      53.15      65.03
Qwen2           79.88       5.68      21.96
Llama3.1        83.92      30.77      53.55
---------------------------------------------
평균              82.95      29.87      46.85

----- EXAONE-3.5-2.4B + EXAONE-3.5-7.8B 장르별 AUC -----
장르             →Solar   →Qwen2  →Llama3.1       평균
---------------------------------------------
essay           96.38    90.28      96.56    94.41
abstract        69.85    71.04      70.49    70.46
poetry          91.61    66.15      81.46    79.74


In [ ]:
# 이전 모델 해제
# del m_e1, m_e2
# torch.cuda.empty_cache()
# print("EXAONE 해제 완료")

print("Polyglot-Ko-1.3B 로드 중...")
tok_p1 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-1.3b')
m_p1   = AutoModelForCausalLM.from_pretrained(
    'EleutherAI/polyglot-ko-1.3b',
    torch_dtype=torch.float16,
    device_map="auto")
m_p1.eval()

print("Polyglot-Ko-3.8B 로드 중...")
tok_p2 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-3.8b')
m_p2   = AutoModelForCausalLM.from_pretrained(
    'EleutherAI/polyglot-ko-3.8b',
    torch_dtype=torch.float16,
    device_map="auto")
m_p2.eval()

result_polyglot = run_ood_experiment(
    "Polyglot-Ko 1.3B + 3.8B",
    m_p1, tok_p1, m_p2, tok_p2,
    human_test, llm_groups, device
)

Polyglot-Ko-1.3B 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Qwen2 점수 계산 중...


100%|██████████| 387/387 [00:28<00:00, 13.54it/s]


Llama3.1 점수 계산 중...


100%|██████████| 338/338 [00:24<00:00, 13.76it/s]



----- Polyglot-Ko 1.3B + 3.8B 전체 결과 -----
LLM               AUC    R@FPR5%   R@FPR10%
---------------------------------------------
Solar           69.06      41.03      47.79
Qwen2           64.56      28.68      44.19
Llama3.1        62.17      26.63      34.32
---------------------------------------------
평균              65.27      32.11      42.10

----- Polyglot-Ko 1.3B + 3.8B 장르별 AUC -----
장르             →Solar   →Qwen2  →Llama3.1       평균
---------------------------------------------
essay           99.72    92.00      98.14    96.62
abstract        74.77    65.16      60.03    66.65
poetry          74.89    74.89      77.01    75.60
